In [1]:
#imports
#python
import os, sys
import scipy
from datetime import datetime
import re
from collections import defaultdict, OrderedDict
import itertools
# image and data processing
import torch
from torchvision.models import alexnet
from torchvision import transforms as T
from PIL import Image , ImageOps
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from sklearn.metrics.pairwise import cosine_similarity
from scipy.optimize import linear_sum_assignment
import scipy.optimize as optimize
from scipy.linalg import block_diag
from scipy.spatial.distance import cdist
from munkres import Munkres, print_matrix
#qiskit and d-wave
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo



In [2]:
import os

def path_joiner_for_category(category_name: str, root_dir: str = None) -> list:
    """
    Finds the directory matching the given category name and returns all .mat files inside it.

    Args:
        category_name (str): e.g., 'car(G)', 'duck(S)', etc.
        root_dir (str): Root directory to start the search from (defaults to cwd).

    Returns:
        list: Full paths to all .mat files inside the category folder.
    """
    if root_dir is None:
        root_dir = os.getcwd()

    for dirpath, dirnames, filenames in os.walk(root_dir):
        if os.path.basename(dirpath) == category_name:
            mat_files = [os.path.join(dirpath, f) for f in filenames if f.endswith('.mat')]
            return sorted(mat_files)  # Sort by name for consistency

    raise FileNotFoundError(f"Category folder '{category_name}' not found in '{root_dir}'")


In [3]:
import scipy.io
import numpy as np

def load_first_four_keypoints(mat_file_paths: list, key_name: str = 'pts_coord') -> dict:
    """
    Loads the first 4 keypoints from each .mat file.

    Args:
        mat_file_paths (list): List of full paths to .mat annotation files.
        key_name (str): Name of the variable inside .mat containing keypoints.

    Returns:
        dict: Mapping of image index (int) to a (4, 2) numpy array of keypoints.
    """
    keypoints_dict = {}

    for idx, path in enumerate(sorted(mat_file_paths)):
        data = scipy.io.loadmat(path)
        keypoints = data[key_name]  # Expecting shape (2, N)
        
        if keypoints.shape[1] < 4:
            raise ValueError(f"File {path} has fewer than 4 keypoints.")

        # Take first 4 keypoints and transpose to shape (4, 2)
        keypoints_dict[idx] = keypoints[:, :4].T

    return keypoints_dict


In [4]:
category = "car(G)"
mat_paths = path_joiner_for_category(category)
keypoints_by_image = load_first_four_keypoints(mat_paths)
print(len(mat_paths))
print(keypoints_by_image)

10
{0: array([[ 47.93528064,  98.15864834],
       [151.86426117, 121.80126002],
       [ 45.965063  ,  57.2766323 ],
       [ 69.60767468,  54.81386025]]), 1: array([[ 38.44802   , 159.21073186],
       [128.33906116, 177.73373428],
       [ 24.28337108, 115.62719675],
       [ 50.43349215, 108.00007811]]), 2: array([[ 24.22058824, 167.35845588],
       [117.125     , 173.22610294],
       [ 14.44117647, 143.39889706],
       [ 24.22058824, 128.72977941]]), 3: array([[ 86.01382619, 160.2722912 ],
       [184.56687359, 183.73730248],
       [ 78.63910835, 134.79599323],
       [ 92.04768623, 124.06913093]]), 4: array([[ 60.57797165, 114.96728462],
       [179.33478735, 151.77208288],
       [ 55.67066521,  86.99563795],
       [ 81.18865867,  79.6346783 ]]), 5: array([[ 60.91071429, 122.78571429],
       [138.58928571, 136.17857143],
       [ 51.80357143,  88.5       ],
       [ 68.41071429,  75.10714286]]), 6: array([[ 81.56511331, 142.3484049 ],
       [193.5711912 , 180.92534969],
 

In [5]:
# generating random subproblems
import random
from itertools import combinations

def generate_4_image_subsets(total_images=40, num_samples=35, seed=42):
    """
    Randomly generates 35 subsets of 4 distinct image indices from total_images.

    Args:
        total_images (int): Total images available (usually 40).
        num_samples (int): Number of 4-image problems to generate.
        seed (int): For reproducibility.

    Returns:
        list of list: Each inner list contains 4 unique image indices.
    """
    random.seed(seed)
    all_combos = list(combinations(range(total_images), 4))
    return random.sample(all_combos, num_samples)


In [6]:
#AlexNet
import torch
import torchvision.transforms as T
from torchvision.models import alexnet
from PIL import Image, ImageOps
import numpy as np
import os

def alexnet_feature_extractor(layer='conv4'):
    model = alexnet(pretrained=True)
    model.eval()
    if layer == 'conv4':
        return torch.nn.Sequential(*list(model.features)[:10])
    elif layer == 'conv5':
        return torch.nn.Sequential(*list(model.features)[:12])
    else:
        raise ValueError("Invalid layer. Choose 'conv4' or 'conv5'.")

def path_joiner_from_mat(mat_path: str, image_ext=".png") -> str:
    base_name = os.path.splitext(os.path.basename(mat_path))[0]
    image_name = base_name + image_ext
    dir_path = os.path.dirname(mat_path)
    image_path = os.path.join(dir_path, image_name)
    if os.path.isfile(image_path):
        return image_path
    for root, _, files in os.walk(dir_path):
        if image_name in files:
            return os.path.join(root, image_name)
    raise FileNotFoundError(f"Image file '{image_name}' not found near '{mat_path}'")

def extract_features_for_indices(keypoints_dict, image_indices, mat_file_paths, 
                                 patch_size=64, layer='conv4'):
    feature_extractor = alexnet_feature_extractor(layer)
    transform = T.Compose([
        T.Resize((227, 227)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    feature_extractor.to(device)
    feature_extractor.eval()

    features = {}
    for idx in image_indices:
        mat_path = mat_file_paths[idx]
        img_path = path_joiner_from_mat(mat_path)
        img = Image.open(img_path).convert('RGB')
        keypoints = keypoints_dict[idx]
        feature_list = []
        for (x, y) in keypoints:
            half = patch_size // 2
            padded_img = ImageOps.expand(img, border=(half, half, half, half), fill=0)
            x_padded = x + half
            y_padded = y + half
            patch = padded_img.crop((x_padded - half, y_padded - half, x_padded + half, y_padded + half))
            patch_tensor = transform(patch).unsqueeze(0).to(device)
            with torch.no_grad():
                feat = feature_extractor(patch_tensor)
                feat = feat.mean(dim=[2, 3])
            feature_list.append(feat.squeeze().cpu().numpy())
        features[idx] = np.stack(feature_list)
    return features



In [9]:
image_subsets = generate_4_image_subsets(total_images=10, num_samples=35, seed=42)
print("Generated image subsets:")
for i, subset in enumerate(image_subsets):
    print(f"Subset {i+1}: {subset}")

# Extract features for the first subset
first_subset = image_subsets[0]
print(f"\nExtracting features for subset: {first_subset}")

# Extract features using AlexNet
# Process all subsets
all_features = {}
for subset_idx, subset in enumerate(image_subsets):
    print(f"\nProcessing subset {subset_idx + 1}: {subset}")
    
    # Extract features using AlexNet
    features = extract_features_for_indices(
        keypoints_by_image,
        subset,
        mat_paths,
        patch_size=64,
        layer='conv4'
    )
    
    # Store features for this subset
    all_features[subset_idx] = features
    
    # Print feature information
    print(f"\nFeature extraction results for subset {subset_idx + 1}:")
    for img_idx, feat in features.items():
        print(f"Image {img_idx} features shape: {feat.shape}")
        print(f"Number of keypoints: {len(feat)}")
        print(f"Feature dimension: {feat.shape[1]}")
        print("-" * 50)

# Print summary
print("\nSummary:")
print(f"Total number of subsets processed: {len(all_features)}")
for subset_idx, features in all_features.items():
    print(f"Subset {subset_idx + 1} contains features for {len(features)} images")

Generated image subsets:
Subset 1: (2, 4, 7, 9)
Subset 2: (0, 2, 3, 4)
Subset 3: (0, 1, 2, 9)
Subset 4: (3, 5, 7, 9)
Subset 5: (0, 4, 6, 9)
Subset 6: (0, 3, 7, 9)
Subset 7: (0, 3, 5, 9)
Subset 8: (0, 2, 4, 6)
Subset 9: (3, 5, 7, 8)
Subset 10: (0, 1, 7, 9)
Subset 11: (2, 6, 8, 9)
Subset 12: (5, 6, 7, 9)
Subset 13: (1, 7, 8, 9)
Subset 14: (0, 1, 6, 7)
Subset 15: (2, 3, 6, 9)
Subset 16: (1, 3, 4, 8)
Subset 17: (0, 1, 3, 5)
Subset 18: (0, 1, 3, 4)
Subset 19: (0, 1, 6, 8)
Subset 20: (0, 3, 5, 7)
Subset 21: (0, 3, 6, 8)
Subset 22: (1, 4, 8, 9)
Subset 23: (2, 3, 8, 9)
Subset 24: (5, 6, 8, 9)
Subset 25: (2, 3, 4, 8)
Subset 26: (0, 3, 4, 6)
Subset 27: (3, 4, 7, 9)
Subset 28: (2, 5, 6, 8)
Subset 29: (3, 4, 6, 7)
Subset 30: (4, 5, 6, 9)
Subset 31: (1, 3, 4, 7)
Subset 32: (0, 3, 5, 8)
Subset 33: (1, 3, 6, 7)
Subset 34: (2, 3, 6, 8)
Subset 35: (0, 4, 7, 8)

Extracting features for subset: (2, 4, 7, 9)

Processing subset 1: (2, 4, 7, 9)

Feature extraction results for subset 1:
Image 2 features shap

In [13]:
def pairwise_permutations(features_dict: dict, pm_size: int) -> dict:
    """
    Computes pairwise permutation matrices P_ij using cosine similarity of keypoint features.
    Adapted to work with the nested dictionary structure of all_features.

    Args:
        features_dict (dict): Dictionary where each key is a subset index and value is another dict
                            containing image indices and their feature matrices (4 x D).
        pm_size (int): Permutation matrix size (number of keypoints, usually 4).

    Returns:
        dict: Dictionary where keys are subset indices and values are dictionaries of 
              pairwise permutation matrices P_ij and P_ji for that subset.
    """
    all_subset_permutations = {}
    
    # Process each subset
    for subset_idx, subset_features in features_dict.items():
        P = {}
        image_list = list(subset_features.keys())
        
        # Generate permutation matrices for this subset
        for i in range(len(image_list)):
            # Identity matrix for same image
            key0 = f'P{i+1}{i+1}'
            P[key0] = np.eye(pm_size, dtype=int)
            
            # Generate permutation matrices between different images
            for j in range(i + 1, len(image_list)):
                img1 = image_list[i]
                img2 = image_list[j]
                
                feats1 = subset_features[img1]
                feats2 = subset_features[img2]
                
                # Compute similarity and cost matrices
                sim_matrix = cosine_similarity(feats1, feats2)
                cost_matrix = 1 - sim_matrix
                
                # Find optimal assignment
                row_ind, col_ind = linear_sum_assignment(cost_matrix)
                perm = np.zeros((pm_size, pm_size), dtype=int)
                perm[row_ind, col_ind] = 1
                
                # Store both P_ij and P_ji
                P[f'P{i+1}{j+1}'] = perm
                P[f'P{j+1}{i+1}'] = perm.T
        
        # Store permutations for this subset
        all_subset_permutations[subset_idx] = P
    
    return all_subset_permutations

In [14]:
perm_matrices = pairwise_permutations(all_features,4)
print(perm_matrices)

{0: {'P11': array([[1, 0, 0, 0],
       [0, 1, 0, 0],
       [0, 0, 1, 0],
       [0, 0, 0, 1]]), 'P12': array([[0, 1, 0, 0],
       [1, 0, 0, 0],
       [0, 0, 0, 1],
       [0, 0, 1, 0]]), 'P21': array([[0, 1, 0, 0],
       [1, 0, 0, 0],
       [0, 0, 0, 1],
       [0, 0, 1, 0]]), 'P13': array([[0, 1, 0, 0],
       [0, 0, 0, 1],
       [1, 0, 0, 0],
       [0, 0, 1, 0]]), 'P31': array([[0, 0, 1, 0],
       [1, 0, 0, 0],
       [0, 0, 0, 1],
       [0, 1, 0, 0]]), 'P14': array([[0, 1, 0, 0],
       [1, 0, 0, 0],
       [0, 0, 1, 0],
       [0, 0, 0, 1]]), 'P41': array([[0, 1, 0, 0],
       [1, 0, 0, 0],
       [0, 0, 1, 0],
       [0, 0, 0, 1]]), 'P22': array([[1, 0, 0, 0],
       [0, 1, 0, 0],
       [0, 0, 1, 0],
       [0, 0, 0, 1]]), 'P23': array([[1, 0, 0, 0],
       [0, 1, 0, 0],
       [0, 0, 0, 1],
       [0, 0, 1, 0]]), 'P32': array([[1, 0, 0, 0],
       [0, 1, 0, 0],
       [0, 0, 0, 1],
       [0, 0, 1, 0]]), 'P24': array([[1, 0, 0, 0],
       [0, 1, 0, 0],
       [0, 0, 1,

In [18]:
import re
import numpy as np

def qubo_form_maker(
    P: dict,
    num_keypoints: int,
    num_views: int = None,
    penalty: float = 2.5
) -> tuple[np.ndarray, np.ndarray]:
    """
    Same as before, but now handles P-keys that are either:
      • strings like "P12", "P3_4", etc.
      • 2-tuples (i,j) or lists [i,j]
    """
    n = num_keypoints

    # infer m if not given
    if num_views is None:
        all_ids = []
        for key in P:
            if isinstance(key, (tuple, list)) and len(key) >= 2:
                all_ids += [key[0], key[1]]
            elif isinstance(key, str):
                nums = re.findall(r'\d+', key)
                if len(nums) >= 2:
                    all_ids += [int(nums[0]), int(nums[1])]
        if not all_ids:
            raise ValueError("Cannot infer num_views from P keys.")
        m = max(all_ids)
    else:
        m = num_views

    if m <= 1:
        return np.zeros((0,0)), np.zeros((0,))

    m_opt = m - 1
    N = n * n
    num_vars = m_opt * N

    Qp = np.zeros((num_vars, num_vars))
    s = np.zeros(num_vars)

    # vec(I)
    x1 = np.zeros(N)
    for k in range(n):
        x1[k*n + k] = 1

    for key, Pij in P.items():
        # parse (i,j) in 0-based
        if isinstance(key, (tuple, list)) and len(key) >= 2:
            i, j = key[0]-1, key[1]-1
        elif isinstance(key, str):
            nums = re.findall(r'\d+', key)
            if len(nums) < 2:
                continue
            i, j = int(nums[0]) - 1, int(nums[1]) - 1
        else:
            continue

        B = np.kron(np.eye(n), Pij)

        if   i > 0 and j > 0:
            oi, oj = i-1, j-1
            r0, r1 = oi*N, (oi+1)*N
            c0, c1 = oj*N, (oj+1)*N
            Qp[r0:r1, c0:c1] -= B

        elif i == 0 and j > 0:
            oj = j-1
            c0, c1 = oj*N, (oj+1)*N
            lin = - x1 @ B
            s[c0:c1] += lin

        elif i > 0 and j == 0:
            oi = i-1
            r0, r1 = oi*N, (oi+1)*N
            lin = - (B @ x1)
            s[r0:r1] += lin

    # build constraint A x = b
    n_cons = m_opt * 2 * n
    A = np.zeros((n_cons, num_vars))
    b = np.ones(n_cons)

    row = 0
    for vi in range(m_opt):
        off = vi * N
        # row‐sum = 1
        for rr in range(n):
            for cc in range(n):
                A[row, off + rr*n + cc] = 1
            row += 1
        # col‐sum = 1
        for cc in range(n):
            for rr in range(n):
                A[row, off + rr*n + cc] = 1
            row += 1

    Q = Qp + penalty * (A.T @ A)
    s = s - 2 * penalty * (A.T @ b)

    # symmetrize
    Q = (Q + Q.T) / 2.0

    return Q, s


In [19]:
Q, s = qubo_form_maker(perm_matrices, num_keypoints=4, num_views=4, penalty=5.0)

In [20]:
print(len(s))
print(Q)
print(s)

48
[[10.  5.  5. ...  0.  0.  0.]
 [ 5. 10.  5. ...  0.  0.  0.]
 [ 5.  5. 10. ...  0.  0.  0.]
 ...
 [ 0.  0.  0. ... 10.  5.  5.]
 [ 0.  0.  0. ...  5. 10.  5.]
 [ 0.  0.  0. ...  5.  5. 10.]]
[-20. -20. -20. -20. -20. -20. -20. -20. -20. -20. -20. -20. -20. -20.
 -20. -20. -20. -20. -20. -20. -20. -20. -20. -20. -20. -20. -20. -20.
 -20. -20. -20. -20. -20. -20. -20. -20. -20. -20. -20. -20. -20. -20.
 -20. -20. -20. -20. -20. -20.]


In [22]:
# from __future__ import annotations

import scipy.optimize as opt
from blueqat import Circuit
from blueqat.pauli import qubo_bit as q

In [23]:
#Using blueQat instead
def _to_float(x):
    """Return *scalar* float even if x is a 0‑D/1‑D NumPy number."""
    return float(np.asarray(x))


def build_blueqat_hamiltonian(Q: np.ndarray, s: np.ndarray):
    """Convert ``cost = xᵀ Q x + sᵀ x`` to a Blueqat PauliSum.

    * Diagonal of *Q* is absorbed into the linear part because *x_i² = x_i*.
    * Off‑diagonal is taken from upper‑triangle only.
    * All coefficients are cast to plain Python ``float`` so that Blueqat’s
      PauliTerm constructor never receives a NumPy scalar (this was the source
      of the previous ValueError).
    """
    Q = np.asarray(Q, dtype=float)
    s = np.asarray(s, dtype=float).ravel()
    n = Q.shape[0]

    H = 0  # PauliSum accumulator ------------------------------------------------

    # linear (includes the diagonal of Q)
    for i in range(n):
        coeff = _to_float(s[i] + Q[i, i])
        if coeff:
            H += coeff * q(i)

    # quadratic (strictly upper‑triangle)
    for i in range(n):
        for j in range(i + 1, n):
            coeff = _to_float(Q[i, j])
            if coeff:
                H += coeff * q(i) * q(j)

    return H.simplify()

In [24]:
H = build_blueqat_hamiltonian(Q,s)
print(H)

-60.0*I + 1.25*Z[0]*Z[12] + 1.25*Z[0]*Z[1] + 1.25*Z[0]*Z[2] + 1.25*Z[0]*Z[3] + 1.25*Z[0]*Z[4] + 1.25*Z[0]*Z[8] - 2.5*Z[0] + 1.25*Z[10]*Z[11] + 1.25*Z[10]*Z[14] - 2.5*Z[10] + 1.25*Z[11]*Z[15] - 2.5*Z[11] + 1.25*Z[12]*Z[13] + 1.25*Z[12]*Z[14] + 1.25*Z[12]*Z[15] - 2.5*Z[12] + 1.25*Z[13]*Z[14] + 1.25*Z[13]*Z[15] - 2.5*Z[13] + 1.25*Z[14]*Z[15] - 2.5*Z[14] - 2.5*Z[15] + 1.25*Z[16]*Z[17] + 1.25*Z[16]*Z[18] + 1.25*Z[16]*Z[19] + 1.25*Z[16]*Z[20] + 1.25*Z[16]*Z[24] + 1.25*Z[16]*Z[28] - 2.5*Z[16] + 1.25*Z[17]*Z[18] + 1.25*Z[17]*Z[19] + 1.25*Z[17]*Z[21] + 1.25*Z[17]*Z[25] + 1.25*Z[17]*Z[29] - 2.5*Z[17] + 1.25*Z[18]*Z[19] + 1.25*Z[18]*Z[22] + 1.25*Z[18]*Z[26] + 1.25*Z[18]*Z[30] - 2.5*Z[18] + 1.25*Z[19]*Z[23] + 1.25*Z[19]*Z[27] + 1.25*Z[19]*Z[31] - 2.5*Z[19] + 1.25*Z[1]*Z[13] + 1.25*Z[1]*Z[2] + 1.25*Z[1]*Z[3] + 1.25*Z[1]*Z[5] + 1.25*Z[1]*Z[9] - 2.5*Z[1] + 1.25*Z[20]*Z[21] + 1.25*Z[20]*Z[22] + 1.25*Z[20]*Z[23] + 1.25*Z[20]*Z[24] + 1.25*Z[20]*Z[28] - 2.5*Z[20] + 1.25*Z[21]*Z[22] + 1.25*Z[21]*Z[23] + 1

In [25]:
# -----------------------------------------------------------------------------
# 2.  Expectation builder
# -----------------------------------------------------------------------------

def make_expectation(H, time_evolutions, p: int, backend="quimb"):
    """Return ``f(params)`` that computes ⟨ψ(β⃗,γ⃗)|H|ψ(β⃗,γ⃗)⟩."""
    n_qubits = H.n_qubits

    def f(params):
        beta = params[:p]
        gamma = params[p:]
        print(f"Evaluating with beta={beta}, gamma={gamma}")

        c = Circuit(n_qubits).h[:]
        for layer in range(p):
            for evo in time_evolutions:
                evo(c, gamma[layer])
            c.rx(beta[layer])[:]
        
        result = c.run(backend=backend, hamiltonian=H)
        print(f"Expectation value: {result}")
        return result

    return f

In [26]:
def run_qaoa(
    Q: np.ndarray,
    s: np.ndarray,
    p: int = 1,
    method: str = "COBYLA",
    tol: float | None = 1e-3,
    random_seed: int | None = None,
    backend: str = "quimb",
):
    """Optimise p‑layer QAOA.

    Returns ``(best_expectation, scipy_result)``.
    """
    print("Starting QAOA optimization...")
    
    if random_seed is not None:
        print("Setting random seed...")
        np.random.seed(random_seed)

    print("Building Hamiltonian...")
    H = build_blueqat_hamiltonian(Q, s)

    print("Calculating time evolutions...")
    time_evos = [t.get_time_evolution() for t in H]

    print("Preparing expectation function...")
    f = make_expectation(H, time_evos, p, backend=backend)

    print("Initializing angles and starting optimization...")
    init = 2 * np.pi * np.random.rand(2 * p)
    res = opt.minimize(f, init, method=method, tol=tol)

    if not res.success:
        raise RuntimeError(res.message)

    print("Optimization complete.")
    return res.fun, res


In [27]:
Eopt, result = run_qaoa(Q, s, p=1, random_seed=42)
print("Optimised ⟨H⟩:", Eopt)
print("β, γ:", result.x)

Starting QAOA optimization...
Setting random seed...
Building Hamiltonian...
Calculating time evolutions...
Preparing expectation function...
Initializing angles and starting optimization...
Evaluating with beta=[2.35330497], gamma=[5.97351416]
Expectation value: -43.111055427920235
Evaluating with beta=[3.35330497], gamma=[5.97351416]
Expectation value: -62.62011820795723
Evaluating with beta=[3.35330497], gamma=[6.97351416]
Expectation value: -60.0061974654802
Evaluating with beta=[4.34444806], gamma=[5.84071591]
Expectation value: -61.195058562608814
Evaluating with beta=[3.84887652], gamma=[5.90711503]
Expectation value: -64.52191071527015
Evaluating with beta=[4.24896407], gamma=[5.60723181]
Expectation value: -59.996608548066185
Evaluating with beta=[3.98810605], gamma=[6.38733908]
Expectation value: -31.724667529441078
Evaluating with beta=[3.8297955], gamma=[5.65784427]
Expectation value: -60.000000004035186
Evaluating with beta=[3.89029837], gamma=[6.02505243]
Expectation valu

In [28]:
print(Eopt)
print(result)

-65.34703224883201
 message: Optimization terminated successfully.
 success: True
  status: 1
     fun: -65.34703224883201
       x: [ 3.827e+00  5.946e+00]
    nfev: 69
   maxcv: 0.0


In [29]:
# H = build_blueqat_hamiltonian(Q, s)

# 2.  Get the list of callables that apply e^{-i γ H_j} for each term
time_evos = [term.get_time_evolution() for term in H]
print(time_evos)

[<function Term.get_time_evolution.<locals>.append_to_circuit at 0x0000022CA9B12B00>, <function Term.get_time_evolution.<locals>.append_to_circuit at 0x0000022CA9B12950>, <function Term.get_time_evolution.<locals>.append_to_circuit at 0x0000022CA9B123B0>, <function Term.get_time_evolution.<locals>.append_to_circuit at 0x0000022CA9B127A0>, <function Term.get_time_evolution.<locals>.append_to_circuit at 0x0000022CA9B12830>, <function Term.get_time_evolution.<locals>.append_to_circuit at 0x0000022CA9B12710>, <function Term.get_time_evolution.<locals>.append_to_circuit at 0x0000022CA9B12680>, <function Term.get_time_evolution.<locals>.append_to_circuit at 0x0000022CA9B13010>, <function Term.get_time_evolution.<locals>.append_to_circuit at 0x0000022CA9B12F80>, <function Term.get_time_evolution.<locals>.append_to_circuit at 0x0000022CA9B12EF0>, <function Term.get_time_evolution.<locals>.append_to_circuit at 0x0000022CA9B12E60>, <function Term.get_time_evolution.<locals>.append_to_circuit at 

In [ ]:
from blueqat import Circuit

n = Q.shape[0]                       # number of binary variables

def make_circuit(betas, gammas):
    c = Circuit(n).h[:]              # start in |+⟩⊗ⁿ
    for β, γ in zip(betas, gammas):
        for evo in time_evos:        # the time_evos list you built earlier
            evo(c, γ)                # cost unitary
        c.rx(β)[:]                  # mixer
    return c.m[:]

# best angles from the optimiser
betas  = [3.82719232]
gammas = [5.94641853]

shots = 1024
counts = make_circuit(betas, gammas).run(shots=shots, backend="quimb")


In [ ]:
print(counts)